# Rubric Agent

This notebook walks through the three-stage rubric pipeline: extracting
requirements from policy text, refining with historical audit findings,
and producing evidence-backed verdicts against a project description.

**IMPORTANT:** an Ollama server must be running locally (`ollama serve`) with the embedding model configured in `config.yaml` (`embeddings.default.model_name`).

## Imports

In [ ]:
from agentic_patterns.agents.rubric.evaluator import RubricEvaluator
from agentic_patterns.agents.rubric.listener import (
    PrintRubricListener,
    PrintRubricEvaluatorListener,
)
from agentic_patterns.agents.rubric.session import RubricSession
from agentic_patterns.core.vectordb import get_vector_db, MultiSourceRetriever
from agentic_patterns.core.vectordb.chunking import chunk_by_paragraphs
from agentic_patterns.core.doc_ingestion.models import DocumentProvenance

## Data

A simplified SOC 2 subset covering access control, encryption, logging, and incident response, plus audit findings from three past reviews.

In [ ]:
POLICY_TEXT = """\
Access Control Policy
All production systems MUST enforce role-based access control (RBAC).
User accounts MUST be reviewed quarterly and inactive accounts disabled within 30 days.
Privileged access SHOULD require multi-factor authentication at every login.

Encryption Policy
Data at rest MUST be encrypted using AES-256 or equivalent.
Data in transit MUST use TLS 1.2 or higher for all internal and external communications.
Encryption keys SHOULD be rotated annually and MAY be rotated more frequently for sensitive systems.

Logging and Monitoring Policy
All authentication events MUST be logged with timestamp, user ID, and outcome.
Administrative actions MUST be captured in an immutable audit trail.
Anomaly detection SHOULD be enabled on critical systems and alerts reviewed within 24 hours.

Incident Response Policy
A documented incident response plan MUST be maintained and tested annually.
Security incidents MUST be reported to the security team within one hour of detection.
Post-incident reviews SHOULD be completed within five business days.
"""

In [ ]:
AUDIT_FINDINGS_TEXT = """\
Q3 Audit Finding: Three service accounts with production access had not been reviewed in over six months.
Q3 Audit Finding: Two internal microservices communicated over plain HTTP instead of TLS.
Q3 Audit Finding: Admin console actions were logged but logs lacked immutability guarantees.
Q3 Audit Finding: A third-party vendor retained SSH access to a production database server for 90 days after project completion.

Q1 Audit Finding: Quarterly access review was completed 15 days late for the payments team.
Q1 Audit Finding: Encryption key rotation had not occurred for the analytics database in 18 months.
Q1 Audit Finding: Two contractor accounts had standing access to the payments service with no defined expiry date.

Q4 Audit Finding: Incident response plan existed but had not been tested since initial creation two years ago.
Q4 Audit Finding: Two critical alerts from the anomaly detection system went unacknowledged for 48 hours.
Q4 Audit Finding: No formal offboarding procedure was followed when an external consultant's engagement ended; access was revoked 47 days later.
"""

## Workflow 1: Incremental

Each `add_document()` runs the full pipeline against one document, then folds results into the existing rubric.

**First call** -- `add_document(POLICY_TEXT)` on an empty rubric:

1. **Chunk** the text by paragraph.
2. **Extract** -- send each chunk to an LLM; it returns structured MUST/SHOULD/MAY requirements as `PoolItem`s.
3. **Merge** -- while pool size > `batch_size`: cluster by semantic similarity, merge each cluster into one item, eject outliers back. Repeat until the pool fits or stops shrinking.
4. **Synthesize** -- process the reduced pool in sequential batches. The agent has three tools (`find_similar`, `add_item`, `add_source`) backed by a vector index. Since the rubric is empty, every item becomes a new `RubricItem`.

**Second call** -- `add_document(AUDIT_FINDINGS_TEXT)` on the rubric from above:

1. Same extract + merge as before, but over the audit text.
2. Synthesize against the **existing rubric**. For each pool item the agent searches for a match:
   - Match found (e.g. "access review was late" ~ quarterly-review requirement) --> `add_source` to the existing item.
   - No match (e.g. vendor/contractor offboarding, absent from policy) --> `add_item` creates a new rubric item.

In [ ]:
session = RubricSession("soc2_demo", listener=PrintRubricListener())
rubric = await session.add_document(POLICY_TEXT, source="soc2_policy")
rubric = await session.add_document(AUDIT_FINDINGS_TEXT, source="audit_findings")

In [ ]:
print(f"Rubric: {rubric.rubric_id}  ({len(rubric.items)} items)\n")
for item in rubric.items:
    n_sources = len(item.sources)
    print(f"[{item.requirement_level.value}] {item.title}  ({n_sources} sources)")

## Workflow 2: Batch

All documents are extracted independently first, then merged and synthesized in a single pass.

**First call** -- `extract(POLICY_TEXT)`:

1. **Chunk** the policy text by paragraph.
2. **Extract** -- send each chunk to an LLM; it returns structured MUST/SHOULD/MAY requirements as `PoolItem`s.
3. **Checkpoint** -- save the pool items to disk. No merging or synthesis happens yet.

**Second call** -- `extract(AUDIT_FINDINGS_TEXT)`:

1. Same chunk + extract as above, but over the audit text.
2. **Checkpoint** -- save the pool items to disk alongside the previous checkpoint.

**`build()`** -- collect all checkpoints and produce the rubric in one shot:

1. **Combine** -- load pool items from both checkpoints into a single pool (policy requirements + audit findings together).
2. **Merge** -- while pool size > `batch_size`: cluster the combined pool by semantic similarity, merge each cluster into one item, eject outliers back. Repeat until the pool fits or stops shrinking. Because all documents are pooled together, cross-document duplicates (e.g. three audit findings about vendor access from different quarters) collapse into one pool item at this stage.
3. **Synthesize** -- process the reduced pool in sequential batches against an **empty rubric**. The agent uses the same three tools (`find_similar`, `add_item`, `add_source`) as in the incremental workflow, but sees the complete picture from all documents in a single pass.

The practical difference: incremental builds the rubric progressively (each document's synthesis sees items from previous documents), while batch pools everything first and builds in one shot. Batch tends to produce a more compact rubric when documents have heavy overlap; incremental is better when you want to see the rubric evolve document by document.

In [ ]:
session_batch = RubricSession("soc2_demo_batch", listener=PrintRubricListener())
await session_batch.extract(POLICY_TEXT, source="soc2_policy")
await session_batch.extract(AUDIT_FINDINGS_TEXT, source="audit_findings")
rubric_batch = await session_batch.build()

## Evaluation

Project Aurora is the system being evaluated. For each rubric item the evaluator retrieves evidence from policy, history, and project indexes, and produces a PASS/RISK/FAIL verdict with citations.

In [ ]:
PROJECT_DESCRIPTION_TEXT = """\
Project Aurora -- Security Posture Summary

Aurora enforces RBAC via AWS IAM with quarterly access reviews automated through a custom script.
MFA is required for all human users but not for CI/CD service accounts.

All databases use AES-256 encryption at rest. Internal service-to-service traffic uses mTLS.
Encryption key rotation is handled by AWS KMS with a 365-day rotation policy.

Authentication events are logged to CloudWatch with structured JSON entries.
Admin actions are captured but stored in the same mutable log stream as application logs.

Aurora has a documented incident response runbook. It was last tested eight months ago.
Anomaly detection is not currently enabled; the team relies on manual dashboard reviews.

There is no documented process for revoking vendor or contractor access upon engagement termination.
"""

In [ ]:
policy_index = get_vector_db("rubric_demo_policy")
policy_index.reset()
policy_index.ingest(
    chunk_by_paragraphs(
        POLICY_TEXT, DocumentProvenance(source="soc2_policy"), min_lines=1
    )
)

history_index = get_vector_db("rubric_demo_history")
history_index.reset()
history_index.ingest(
    chunk_by_paragraphs(
        AUDIT_FINDINGS_TEXT, DocumentProvenance(source="audit_findings"), min_lines=1
    )
)

project_index = get_vector_db("rubric_demo_project")
project_index.reset()
project_index.ingest(
    chunk_by_paragraphs(
        PROJECT_DESCRIPTION_TEXT,
        DocumentProvenance(source="project_aurora"),
        min_lines=1,
    )
)

retriever = MultiSourceRetriever(
    policy=policy_index,
    history=history_index,
    project=project_index,
)
evaluator = RubricEvaluator(listener=PrintRubricEvaluatorListener())
verdicts = await evaluator.evaluate(rubric, retriever)

In [ ]:
title_by_id = {item.item_id: item.title for item in rubric.items}
by_status = {"FAIL": [], "RISK": [], "PASS": []}
for v in verdicts:
    by_status[v.status.value].append(v)

for label in ("FAIL", "RISK", "PASS"):
    group = by_status[label]
    if not group:
        continue
    for v in group:
        title = title_by_id.get(v.item_id, v.item_id)
        note = v.rationale.split(".")[0].strip()
        if len(note) > 90:
            note = note[:87] + "..."
        print(f"[{label}]  {title}")
        print(f"       {note}")
    print()